# Silver User Mapping

Extract every `user_32digit` identifier from silver tables into a single mapping.
Email pairing uses **real personal emails only** (not masked/system emails):
- Contacts: name/display_name + email from contacts_emails
- Activities: exploded `attendees` array → contact_id links to contacts for user_32digit + attendees.email (real personal)
- Organizers: attendees.user_id (user_43digit) → close_crm_users for staff email
- Calendly: INVITEE_EMAIL + EVENT_HOST_EMAIL
- mdl_users: EMAIL column
Master user ID groups user_32digit values sharing the same **real** email — masked emails are kept for lookup but excluded from grouping.

In [0]:
# Step 1: Extract user_32digit + email from all silver tables
# DEDUP: all source tables have CDC duplicates — DISTINCT before joins to prevent row multiplication.
# CHANGES from v1:
#   - DITCHED: envelope_from/to (shared/system emails), created_by_name, user_name, updated_by_name (staff actions)
#   - ADDED: exploded attendees array → real personal emails + contact_id link to user_32digit
#   - Activities now contribute identity through attendees, not through staff action columns
# Email sources:
#   - contacts: LEFT JOIN to contacts_emails on cont_id (may be masked or real)
#   - activities attendees: attendees.email (real personal — gmail, outlook, @dea.com)
#   - activities organizers: attendees.user_id → close_crm_users.email (staff)
#   - calendly: INVITEE_EMAIL + EVENT_HOST_EMAIL
#   - mdl_users: EMAIL column

spark.sql("""
WITH contacts_dedup AS (
    SELECT DISTINCT id, leads_processed_id, name, display_name
    FROM crm_ingestion.silver.leads_processed_contacts
),
emails_dedup AS (
    SELECT DISTINCT leads_processed_contacts_id, email
    FROM crm_ingestion.silver.leads_processed_contacts_emails
),
close_crm_dedup AS (
    SELECT DISTINCT id, email
    FROM crm_ingestion.silver.close_crm_users_processed
),
-- Explode attendees array — real personal emails + contact_id link
attendees_dedup AS (
    SELECT DISTINCT
        a.id AS activity_id,
        a.lead_id,
        att.contact_id,
        att.email AS attendee_email,
        att.user_id AS attendee_user_id
    FROM crm_ingestion.silver.lead_activites_processed a
    LATERAL VIEW explode(a.attendees) AS att
    WHERE a.attendees IS NOT NULL AND size(a.attendees) > 0
),
leads_proc AS (
    SELECT DISTINCT LOWER(TRIM(name)) AS user_32digit, 'leads_processed' AS src_table, 'name' AS src_col,
           id AS lead_id, NULL AS cont_id, NULL AS activity_id, NULL AS email
    FROM crm_ingestion.silver.leads_processed
    WHERE name LIKE 'user_%' AND LENGTH(name) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(display_name)), 'leads_processed', 'display_name',
           id, NULL, NULL, NULL
    FROM crm_ingestion.silver.leads_processed
    WHERE display_name LIKE 'user_%' AND LENGTH(display_name) = 37
),
contacts AS (
    SELECT DISTINCT LOWER(TRIM(c.name)) AS user_32digit, 'leads_processed_contacts' AS src_table, 'name' AS src_col,
           c.leads_processed_id AS lead_id, c.id AS cont_id, NULL AS activity_id, ce.email AS email
    FROM contacts_dedup c
    LEFT JOIN emails_dedup ce ON c.id = ce.leads_processed_contacts_id
    WHERE c.name LIKE 'user_%' AND LENGTH(c.name) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(c.display_name)), 'leads_processed_contacts', 'display_name',
           c.leads_processed_id, c.id, NULL, ce.email
    FROM contacts_dedup c
    LEFT JOIN emails_dedup ce ON c.id = ce.leads_processed_contacts_id
    WHERE c.display_name LIKE 'user_%' AND LENGTH(c.display_name) = 37
),
-- Activities via attendees: contact_id → contacts for user_32digit, attendees.email for real personal email
activities AS (
    -- Students: contact_id → contacts.name (user_32digit) + attendees.email (personal)
    SELECT DISTINCT LOWER(TRIM(c.name)) AS user_32digit, 'lead_activites' AS src_table, 'attendees_name' AS src_col,
           att.lead_id, att.contact_id AS cont_id, att.activity_id, att.attendee_email AS email
    FROM attendees_dedup att
    JOIN contacts_dedup c ON att.contact_id = c.id
    WHERE c.name LIKE 'user_%' AND LENGTH(c.name) = 37
    UNION ALL
    -- Students: contact_id → contacts.display_name (user_32digit) + attendees.email
    SELECT DISTINCT LOWER(TRIM(c.display_name)), 'lead_activites', 'attendees_display_name',
           att.lead_id, att.contact_id, att.activity_id, att.attendee_email
    FROM attendees_dedup att
    JOIN contacts_dedup c ON att.contact_id = c.id
    WHERE c.display_name LIKE 'user_%' AND LENGTH(c.display_name) = 37
    UNION ALL
    -- Organizers: attendees.user_id (user_43digit) → close_crm_users.email (staff email)
    SELECT DISTINCT LOWER(TRIM(att.attendee_user_id)), 'lead_activites', 'attendees_user_id',
           att.lead_id, NULL AS cont_id, att.activity_id, cu.email
    FROM attendees_dedup att
    LEFT JOIN close_crm_dedup cu ON att.attendee_user_id = cu.id
    WHERE att.attendee_user_id LIKE 'user_%' AND LENGTH(att.attendee_user_id) = 48
),
calendly AS (
    SELECT DISTINCT LOWER(TRIM(INVITEE_NAME)) AS user_32digit, 'calendly' AS src_table, 'INVITEE_NAME' AS src_col,
           NULL AS lead_id, NULL AS cont_id, NULL AS activity_id, INVITEE_EMAIL AS email
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE INVITEE_NAME LIKE 'user_%' AND LENGTH(INVITEE_NAME) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(EVENT_HOST_NAME)), 'calendly', 'EVENT_HOST_NAME',
           NULL, NULL, NULL, EVENT_HOST_EMAIL
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE EVENT_HOST_NAME LIKE 'user_%' AND LENGTH(EVENT_HOST_NAME) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(PROFILE_NAME)), 'calendly', 'PROFILE_NAME',
           NULL, NULL, NULL, NULL
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE PROFILE_NAME LIKE 'user_%' AND LENGTH(PROFILE_NAME) = 37
),
mdl AS (
    SELECT DISTINCT LOWER(TRIM(FIRSTNAME)) AS user_32digit, 'mdl_users' AS src_table, 'FIRSTNAME' AS src_col,
           NULL AS lead_id, NULL AS cont_id, NULL AS activity_id, EMAIL AS email
    FROM crm_ingestion.silver.mdl_users_processed
    WHERE FIRSTNAME LIKE 'user_%' AND LENGTH(FIRSTNAME) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(LASTNAME)), 'mdl_users', 'LASTNAME',
           NULL, NULL, NULL, EMAIL
    FROM crm_ingestion.silver.mdl_users_processed
    WHERE LASTNAME LIKE 'user_%' AND LENGTH(LASTNAME) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(USERNAME)), 'mdl_users', 'USERNAME',
           NULL, NULL, NULL, EMAIL
    FROM crm_ingestion.silver.mdl_users_processed
    WHERE USERNAME LIKE 'user_%' AND LENGTH(USERNAME) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(LASTIP)), 'mdl_users', 'LASTIP',
           NULL, NULL, NULL, NULL
    FROM crm_ingestion.silver.mdl_users_processed
    WHERE LASTIP LIKE 'user_%' AND LENGTH(LASTIP) = 37
),
sentiment AS (
    SELECT DISTINCT LOWER(TRIM(CHANNEL_ID)) AS user_32digit, 'student_sentiment' AS src_table, 'CHANNEL_ID' AS src_col,
           NULL AS lead_id, NULL AS cont_id, NULL AS activity_id, NULL AS email
    FROM crm_ingestion.silver.student_sentiment_processed
    WHERE CHANNEL_ID LIKE 'user_%' AND LENGTH(CHANNEL_ID) = 37
    UNION ALL
    SELECT DISTINCT LOWER(TRIM(CHANNEL_NAME)), 'student_sentiment', 'CHANNEL_NAME',
           NULL, NULL, NULL, NULL
    FROM crm_ingestion.silver.student_sentiment_processed
    WHERE CHANNEL_NAME LIKE 'user_%' AND LENGTH(CHANNEL_NAME) = 37
)
SELECT * FROM leads_proc
UNION ALL SELECT * FROM contacts
UNION ALL SELECT * FROM activities
UNION ALL SELECT * FROM calendly
UNION ALL SELECT * FROM mdl
UNION ALL SELECT * FROM sentiment
""").createOrReplaceTempView("user_mapping_step1")

# Show counts by source table + column
print("=== Rows by source table + column ===")
spark.sql("""
SELECT src_table, src_col, COUNT(*) AS row_count, COUNT(DISTINCT user_32digit) AS distinct_users,
       SUM(CASE WHEN email IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_email
FROM user_mapping_step1
GROUP BY src_table, src_col
ORDER BY src_table, src_col
""").show(50, truncate=False)

# Total distinct user_32digit
print("=== Total distinct user_32digit across all tables ===")
spark.sql("SELECT COUNT(DISTINCT user_32digit) AS total_distinct FROM user_mapping_step1").show()

# How many user_32digit have at least one email?
print("=== user_32digit with at least one email ===")
spark.sql("SELECT COUNT(DISTINCT user_32digit) AS users_with_email FROM user_mapping_step1 WHERE email IS NOT NULL").show()

# Email type breakdown
print("=== Email type breakdown ===")
spark.sql("""
SELECT 
    CASE WHEN email LIKE '%@masked.com' THEN 'masked' WHEN email IS NOT NULL THEN 'real' ELSE 'no_email' END AS email_type,
    COUNT(DISTINCT user_32digit) AS distinct_users
FROM user_mapping_step1
GROUP BY CASE WHEN email LIKE '%@masked.com' THEN 'masked' WHEN email IS NOT NULL THEN 'real' ELSE 'no_email' END
""").show()

# Sample rows with real email
print("=== Sample rows with real (non-masked) email ===")
spark.sql("SELECT * FROM user_mapping_step1 WHERE email IS NOT NULL AND email NOT LIKE '%@masked.com' LIMIT 20").show(20, truncate=False)

In [0]:
# Step 2: Group by email — one row per email with all linked user_32digit values and context
# This reveals when the same email has different user_32digit hashes across systems (same person, different hash)
# Also preserves lead_ids, cont_ids, and source tables linked to each email

spark.sql("""
SELECT 
    email,
    collect_set(user_32digit) AS user_32digits,
    collect_set(CASE WHEN lead_id IS NOT NULL THEN lead_id END) AS lead_ids,
    collect_set(CASE WHEN cont_id IS NOT NULL THEN cont_id END) AS cont_ids,
    collect_set(src_table) AS source_tables,
    size(collect_set(user_32digit)) AS num_user_32digits,
    size(collect_set(CASE WHEN lead_id IS NOT NULL THEN lead_id END)) AS num_lead_ids,
    size(collect_set(src_table)) AS num_sources
FROM user_mapping_step1
WHERE email IS NOT NULL
GROUP BY email
""").createOrReplaceTempView("user_mapping_step2")

# Summary stats
print("=== Summary ===")
spark.sql("""
SELECT 
    COUNT(*) AS total_emails,
    SUM(CASE WHEN num_user_32digits > 1 THEN 1 ELSE 0 END) AS emails_with_multiple_user_32digits,
    SUM(CASE WHEN num_lead_ids > 0 THEN 1 ELSE 0 END) AS emails_linked_to_leads,
    SUM(CASE WHEN num_sources > 1 THEN 1 ELSE 0 END) AS emails_in_multiple_sources,
    SUM(CASE WHEN email LIKE '%@masked.com' THEN 1 ELSE 0 END) AS masked_emails,
    SUM(CASE WHEN email NOT LIKE '%@masked.com' THEN 1 ELSE 0 END) AS real_emails
FROM user_mapping_step2
""").show()

# Emails with multiple user_32digit values — same person, different hash across systems
print("=== Emails with multiple user_32digit values (cross-system identity) ===")
spark.sql("""
SELECT email, user_32digits, source_tables, num_user_32digits, num_lead_ids
FROM user_mapping_step2
WHERE num_user_32digits > 1
ORDER BY num_user_32digits DESC
LIMIT 20
""").show(20, truncate=False)

# Real (non-masked) emails — these are the ones that bridge systems
print("=== Real (non-masked) emails with linked lead_ids ===")
spark.sql("""
SELECT email, user_32digits, lead_ids, source_tables, num_user_32digits, num_lead_ids
FROM user_mapping_step2
WHERE email NOT LIKE '%@masked.com'
  AND num_lead_ids > 0
ORDER BY num_lead_ids DESC
LIMIT 20
""").show(20, truncate=False)

# Sample grouped rows
print("=== Sample grouped rows ===")
spark.sql("SELECT * FROM user_mapping_step2 LIMIT 20").show(20, truncate=False)

In [0]:
# Step 3: Pivot user_32digit back to original column names
# One row per email, with columns matching each source table's column names
# Each original table joins to this mapping using its own column name:
#   leads_processed_contacts.name = mapping.contacts_name
#   calendly.INVITEE_NAME = mapping.calendly_invitee_name
#   mdl_users.FIRSTNAME = mapping.mdl_firstname
# collect_set ignores NULLs automatically, so missing sources return empty array []

spark.sql("""
SELECT
    email,
    -- leads_processed
    collect_set(CASE WHEN src_table='leads_processed' AND src_col='name' THEN user_32digit END) AS leads_name,
    collect_set(CASE WHEN src_table='leads_processed' AND src_col='display_name' THEN user_32digit END) AS leads_display_name,
    -- leads_processed_contacts
    collect_set(CASE WHEN src_table='leads_processed_contacts' AND src_col='name' THEN user_32digit END) AS contacts_name,
    collect_set(CASE WHEN src_table='leads_processed_contacts' AND src_col='display_name' THEN user_32digit END) AS contacts_display_name,
    -- lead_activites (attendees-based)
    collect_set(CASE WHEN src_table='lead_activites' AND src_col='attendees_name' THEN user_32digit END) AS attendees_name,
    collect_set(CASE WHEN src_table='lead_activites' AND src_col='attendees_display_name' THEN user_32digit END) AS attendees_display_name,
    collect_set(CASE WHEN src_table='lead_activites' AND src_col='attendees_user_id' THEN user_32digit END) AS attendees_user_id,
    -- calendly
    collect_set(CASE WHEN src_table='calendly' AND src_col='INVITEE_NAME' THEN user_32digit END) AS calendly_invitee_name,
    collect_set(CASE WHEN src_table='calendly' AND src_col='EVENT_HOST_NAME' THEN user_32digit END) AS calendly_host_name,
    collect_set(CASE WHEN src_table='calendly' AND src_col='PROFILE_NAME' THEN user_32digit END) AS calendly_profile_name,
    -- mdl_users
    collect_set(CASE WHEN src_table='mdl_users' AND src_col='FIRSTNAME' THEN user_32digit END) AS mdl_firstname,
    collect_set(CASE WHEN src_table='mdl_users' AND src_col='LASTNAME' THEN user_32digit END) AS mdl_lastname,
    collect_set(CASE WHEN src_table='mdl_users' AND src_col='USERNAME' THEN user_32digit END) AS mdl_username,
    collect_set(CASE WHEN src_table='mdl_users' AND src_col='LASTIP' THEN user_32digit END) AS mdl_lastip,
    -- student_sentiment
    collect_set(CASE WHEN src_table='student_sentiment' AND src_col='CHANNEL_ID' THEN user_32digit END) AS sentiment_channel_id,
    collect_set(CASE WHEN src_table='student_sentiment' AND src_col='CHANNEL_NAME' THEN user_32digit END) AS sentiment_channel_name,
    -- Context IDs
    collect_set(CASE WHEN lead_id IS NOT NULL THEN lead_id END) AS lead_ids,
    collect_set(CASE WHEN cont_id IS NOT NULL THEN cont_id END) AS cont_ids,
    -- Source coverage
    collect_set(src_table) AS source_tables
FROM user_mapping_step1
WHERE email IS NOT NULL
GROUP BY email
""").createOrReplaceTempView("user_mapping")

# Summary: how many emails have each column populated
print("=== Pivot Table: Column Coverage ===")
spark.sql("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN size(leads_name) > 0 THEN 1 ELSE 0 END) AS has_leads_name,
    SUM(CASE WHEN size(contacts_name) > 0 THEN 1 ELSE 0 END) AS has_contacts_name,
    SUM(CASE WHEN size(attendees_name) > 0 THEN 1 ELSE 0 END) AS has_attendees_name,
    SUM(CASE WHEN size(attendees_display_name) > 0 THEN 1 ELSE 0 END) AS has_attendees_display_name,
    SUM(CASE WHEN size(attendees_user_id) > 0 THEN 1 ELSE 0 END) AS has_attendees_user_id,
    SUM(CASE WHEN size(calendly_invitee_name) > 0 THEN 1 ELSE 0 END) AS has_calendly_invitee,
    SUM(CASE WHEN size(calendly_host_name) > 0 THEN 1 ELSE 0 END) AS has_calendly_host,
    SUM(CASE WHEN size(mdl_firstname) > 0 THEN 1 ELSE 0 END) AS has_mdl_firstname,
    SUM(CASE WHEN size(mdl_lastname) > 0 THEN 1 ELSE 0 END) AS has_mdl_lastname,
    SUM(CASE WHEN size(mdl_username) > 0 THEN 1 ELSE 0 END) AS has_mdl_username,
    SUM(CASE WHEN size(sentiment_channel_name) > 0 THEN 1 ELSE 0 END) AS has_sentiment,
    SUM(CASE WHEN size(lead_ids) > 0 THEN 1 ELSE 0 END) AS has_lead_id,
    SUM(CASE WHEN email NOT LIKE '%@masked.com' THEN 1 ELSE 0 END) AS real_emails,
    SUM(CASE WHEN email LIKE '%@masked.com' THEN 1 ELSE 0 END) AS masked_emails
FROM user_mapping
""").show()

# Emails that bridge multiple systems
print("=== Emails linking CRM + Calendly + mdl ===")
spark.sql("""
SELECT email, contacts_name, calendly_invitee_name, mdl_firstname, mdl_username, lead_ids, source_tables
FROM user_mapping
WHERE size(contacts_name) > 0
  AND size(calendly_invitee_name) > 0
LIMIT 20
""").show(20, truncate=False)

# Emails linking CRM + mdl (no Calendly)
print("=== Emails linking CRM + mdl only ===")
spark.sql("""
SELECT email, contacts_name, mdl_firstname, mdl_username, lead_ids, source_tables
FROM user_mapping
WHERE size(contacts_name) > 0
  AND size(calendly_invitee_name) = 0
  AND size(mdl_firstname) > 0
LIMIT 20
""").show(20, truncate=False)

# Schema
print("=== Schema ===")
spark.sql("DESCRIBE user_mapping").show(50, truncate=False)

In [0]:
# Step 4: Create master user ID
# Bundle user_32digit values that share the same REAL email into one group.
# Only non-masked emails (gmail, outlook, @dea.com) drive grouping.
# Masked emails (@masked.com) kept in step1 for lookup but excluded from grouping
#   — they're often shared/system emails that falsely bundle different people.
# Each group gets a sequential ID (user_1, user_2, ...).
# For user_32digit without a real email, each is its own group.

spark.sql("""
WITH real_email_counts AS (
    SELECT user_32digit, COUNT(DISTINCT email) AS num_real_emails
    FROM user_mapping_step1
    WHERE email IS NOT NULL AND email NOT LIKE '%@masked.com'
    GROUP BY user_32digit
),
user_email_pairs AS (
    SELECT DISTINCT s.user_32digit, s.email
    FROM user_mapping_step1 s
    LEFT JOIN real_email_counts r ON s.user_32digit = r.user_32digit
    WHERE s.email IS NOT NULL
      AND s.email NOT LIKE '%@masked.com'
      AND COALESCE(r.num_real_emails, 0) <= 10  -- exclude shared/system hashes with >10 real emails
),
-- Self-join on email: user_32digit values sharing any real email get the same group key (MIN user_32digit)
user_groups AS (
    SELECT ue1.user_32digit, MIN(ue2.user_32digit) AS group_key
    FROM user_email_pairs ue1
    JOIN user_email_pairs ue2 ON ue1.email = ue2.email
    GROUP BY ue1.user_32digit
),
-- All user_32digit values with group key (self if no real email)
all_with_groups AS (
    SELECT DISTINCT s.user_32digit,
           COALESCE(g.group_key, s.user_32digit) AS group_key
    FROM (SELECT DISTINCT user_32digit FROM user_mapping_step1) s
    LEFT JOIN user_groups g ON s.user_32digit = g.user_32digit
)
SELECT a.user_32digit,
       CONCAT('user_', DENSE_RANK() OVER (ORDER BY a.group_key)) AS master_user_id,
       a.group_key
FROM all_with_groups a
""").createOrReplaceTempView("user_master_mapping")

# Summary
print("=== Master User ID Summary ===")
spark.sql("""
SELECT 
    COUNT(DISTINCT master_user_id) AS total_master_ids,
    COUNT(DISTINCT user_32digit) AS total_user_32digits
FROM user_master_mapping
""").show()

# Group size distribution
print("=== Group Size Distribution ===")
spark.sql("""
SELECT num_users, COUNT(*) AS num_groups
FROM (
    SELECT master_user_id, COUNT(DISTINCT user_32digit) AS num_users
    FROM user_master_mapping
    GROUP BY master_user_id
)
GROUP BY num_users
ORDER BY num_users
""").show(50, truncate=False)

# Groups with multiple user_32digit values (cross-system identities bundled)
print("=== Groups with multiple user_32digit values ===")
spark.sql("""
SELECT master_user_id, collect_set(user_32digit) AS user_32digits, COUNT(*) AS num_users
FROM user_master_mapping
GROUP BY master_user_id
HAVING COUNT(*) > 1
ORDER BY num_users DESC
LIMIT 20
""").show(20, truncate=False)

# Sample
print("=== Sample master mapping ===")
spark.sql("SELECT * FROM user_master_mapping LIMIT 20").show(20, truncate=False)